# FastAPI streaming agent — SSE + HITL resume

Two endpoints, one Python file. `GET /agent/stream` is SSE — perfect for the user-visible chat. `POST /agent/resume` carries the human's decision. The interrupt-and-resume pattern survives a stateless HTTP front-end because the server owns the checkpoint.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / '07-deployment-patterns' / '00-fastapi-streaming-agent'))
sys.path.insert(0, str(ROOT / '04-human-in-the-loop'))
os.environ.setdefault('LLM_CACHE_ONLY', '1')

from fastapi.testclient import TestClient
from app import app, THREADS
THREADS.clear()
client = TestClient(app)
client.get('/healthz').json()

In [ ]:
import threading, time, json
chunks = []
def _stream():
    with client.stream('GET', '/agent/stream', params={'question': 'Summarise RA-MoE.', 'thread_id': 'demo'}) as r:
        for c in r.iter_text():
            chunks.append(c)
            if 'complete' in c: return
th = threading.Thread(target=_stream, daemon=True); th.start()

# Wait for the interrupt frame.
for _ in range(50):
    if any('interrupt' in c for c in chunks): break
    time.sleep(0.05)
print('frames so far:', len(chunks))
print(chunks[-1][:300])

In [ ]:
# Approve the publish step.
print(client.post('/agent/resume', json={'thread_id': 'demo', 'decision': 'approve'}).json())
th.join(timeout=5)
print('total frames:', len(chunks))

## Production checklist

* Persist checkpoints in Postgres (or Redis) so a pod restart doesn't drop the interrupt.
* Use a per-thread Redis pub/sub to fan-out SSE if you scale horizontally.
* Authenticate the `resume` POST against the same identity that started the stream.